In [1]:
# cuedetat_pocket_detector_kaggle.py
#
# Kaggle-native re-train of the YOLOv8n pocket/pool detector for CueDetat.
# Mirrors the same hyperparameters and class set as the original Colab notebook,
# but stripped of Drive mounts and `/content/` paths so it runs unmodified on
# Kaggle GPU.
#
# Cells delimited with `# %%` for jupytext-style conversion. Convert with:
#   `python ml/_scripts/py_to_ipynb.py ml/cuedetat_pocket_detector_kaggle.py`
#
# Output: /kaggle/working/pocket_detector_fp16.tflite (deploy as
# app/src/main/assets/ml/merged_pocket_detector_final_float16.tflite).

# CueDetat pocket detector — Kaggle re-train

YOLOv8n. 100 epochs, batch 32, imgsz 640, FP16 TFLite export with NMS.
Datasets: `hereliesaz/cue-detat` + `diveshcrazy/pool-table-balls-classification`.

## Cell 1 — Install (skip torch — Kaggle's preinstalled torch matches the GPU)

In [2]:
!pip install -q ultralytics kagglehub pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.8 MB/s eta 0:00:00


## Cell 2 — Attach Kaggle datasets via kagglehub

These also work via the Kaggle UI ("Add Data" → search), but kagglehub is
explicit and survives notebook re-runs.

In [3]:
import os, kagglehub

CUE_PATH   = kagglehub.dataset_download("hereliesaz/cue-detat")
BALLS_PATH = kagglehub.dataset_download("diveshcrazy/pool-table-balls-classification")
print("cue-detat:", CUE_PATH)
print("balls    :", BALLS_PATH)
print("contents (cue-detat):", os.listdir(CUE_PATH)[:10])
print("contents (balls):    ", os.listdir(BALLS_PATH)[:10])

cue-detat: /kaggle/input/datasets/hereliesaz/cue-detat
balls    : /kaggle/input/datasets/diveshcrazy/pool-table-balls-classification
contents (cue-detat): ['Pool Table Detection.v6i.yolov8', 'pool-balls-yolov11m.v14i.yolov8', 'CueTor Billiards Training.v8i.yolov8']
contents (balls):     ['Dataset']


## Cell 3 — Merge into a unified YOLO dataset

Master class list (matches the deployed model's expectations):
  0: pool-table
  1: pool-table-hole  (pockets — what the AR overlay actually consumes)
  2: pool-table-side

Each source is scanned for `train/`, `valid/`, `test/` splits with
`images/` + `labels/` subdirs. Class IDs are remapped per source via its
`data.yaml` -> master class names mapping.

In [4]:
import os, shutil, yaml, glob

MASTER_CLASSES = ["pool-table", "pool-table-hole", "pool-table-side"]
COMBINED = "/kaggle/working/combined_dataset"

for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(COMBINED, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(COMBINED, split, "labels"), exist_ok=True)

def find_data_yaml(root):
    for p in glob.iglob(os.path.join(root, "**", "data.yaml"), recursive=True):
        return p
    return None

def remap_label_file(src_path, dst_path, id_map):
    """Rewrite a YOLO label file remapping class IDs; drop classes not in id_map."""
    out_lines = []
    with open(src_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id not in id_map:
                continue
            new_id = id_map[old_id]
            out_lines.append(" ".join([str(new_id)] + parts[1:]))
    if out_lines:
        with open(dst_path, "w") as f:
            f.write("\n".join(out_lines) + "\n")

def merge_source(src_root, tag):
    yaml_path = find_data_yaml(src_root)
    if not yaml_path:
        print(f"  [{tag}] no data.yaml found, skipping")
        return 0
    with open(yaml_path) as f:
        cfg = yaml.safe_load(f)
    src_names = cfg.get("names") or []
    # Build remap: src class id -> master id (if name matches), else drop
    id_map = {i: MASTER_CLASSES.index(n)
              for i, n in enumerate(src_names)
              if n in MASTER_CLASSES}
    if not id_map:
        print(f"  [{tag}] no overlap with master classes "
              f"(src had {src_names}); skipping")
        return 0
    print(f"  [{tag}] class remap: "
          f"{ {src_names[i]: MASTER_CLASSES[id_map[i]] for i in id_map} }")

    src_base = os.path.dirname(yaml_path)
    copied = 0
    for split_in, split_out in [("train", "train"), ("valid", "valid"),
                                ("val", "valid"), ("test", "test")]:
        img_src = os.path.join(src_base, split_in, "images")
        lbl_src = os.path.join(src_base, split_in, "labels")
        if not os.path.isdir(img_src):
            continue
        img_dst = os.path.join(COMBINED, split_out, "images")
        lbl_dst = os.path.join(COMBINED, split_out, "labels")
        for img in os.listdir(img_src):
            stem, ext = os.path.splitext(img)
            new_name = f"{tag}__{stem}"
            shutil.copy2(os.path.join(img_src, img),
                         os.path.join(img_dst, new_name + ext))
            lbl_in = os.path.join(lbl_src, stem + ".txt")
            if os.path.exists(lbl_in):
                remap_label_file(
                    lbl_in,
                    os.path.join(lbl_dst, new_name + ".txt"),
                    id_map,
                )
            copied += 1
    return copied

print("Merging cue-detat...")
n1 = merge_source(CUE_PATH, "cue")
print(f"  copied {n1} images")
print("Merging pool-table-balls...")
n2 = merge_source(BALLS_PATH, "balls")
print(f"  copied {n2} images")

# Write the combined data.yaml
data_yaml = os.path.join(COMBINED, "data.yaml")
with open(data_yaml, "w") as f:
    yaml.safe_dump({
        "path":  COMBINED,
        "train": "train/images",
        "val":   "valid/images",
        "test":  "test/images",
        "nc":    len(MASTER_CLASSES),
        "names": MASTER_CLASSES,
    }, f)

# Quick sanity counts
for split in ["train", "valid", "test"]:
    n = len(os.listdir(os.path.join(COMBINED, split, "images")))
    print(f"  {split}: {n} images")
print(f"data.yaml written: {data_yaml}")

Merging cue-detat...
  [cue] class remap: {'pool-table': 'pool-table', 'pool-table-hole': 'pool-table-hole', 'pool-table-side': 'pool-table-side'}
  copied 3234 images
Merging pool-table-balls...
  [balls] no data.yaml found, skipping
  copied 0 images
  train: 2688 images
  valid: 284 images
  test: 262 images
data.yaml written: /kaggle/working/combined_dataset/data.yaml


## Cell 4 — Train YOLOv8n

Hyperparameters mirror the original training: 100 epochs, batch 32, imgsz 640,
AMP, periodic checkpointing.

In [5]:
import os, torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", device, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT = "/kaggle/working/runs"
NAME    = "pocket_detector"

model = YOLO("yolov8n.pt")
results = model.train(
    data="/kaggle/working/combined_dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=32,
    device=device,
    workers=4,
    cache=True,
    amp=True,
    save_period=10,
    project=PROJECT,
    name=NAME,
    exist_ok=True,
    patience=30,    # early stopping if no improvement
)
print("Training done.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: 0 | CUDA available: True
GPU: Tesla T4
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/combined_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

## Cell 5 — Validate

In [6]:
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"Acceptance gate: mAP50 >= 0.80. "
      f"{'PASS' if metrics.box.map50 >= 0.80 else 'FAIL — investigate before deploying'}")

Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1134.9±302.4 MB/s, size: 46.7 KB)
val: Scanning /kaggle/working/combined_dataset/valid/labels.cache... 284 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 284/284 119.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        284       3584      0.977      0.968      0.991      0.815
            pool-table        283        283      0.991          1      0.994       0.96
       pool-table-hole        284       1626      0.991      0.972      0.993      0.671
       pool-table-side        284       1675      0.951      0.932      0.986      0.815
Speed: 1.7ms preprocess, 4.1ms inference, 0.0ms loss, 7.9ms postprocess per image
Results saved to /kaggle/working/runs/d

## Cell 6 — Export to TFLite FP16 with NMS

In [7]:
import os, shutil

EXPORT_DIR = "/kaggle/working/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

print("Exporting TFLite FP16 with NMS...")
exported = model.export(format="tflite", imgsz=640, half=True, nms=True)
print("Export returned:", exported)

# Locate the actual .tflite file (export may return a dir or a file path)
tflite_src = None
if isinstance(exported, str):
    if exported.endswith(".tflite") and os.path.isfile(exported):
        tflite_src = exported
    elif os.path.isdir(exported):
        cands = [f for f in os.listdir(exported) if f.endswith(".tflite")]
        if cands:
            tflite_src = os.path.join(exported, cands[0])
if tflite_src is None:
    # fallback: scan the runs dir
    for root, _, files in os.walk(PROJECT):
        for f in files:
            if f.endswith(".tflite"):
                tflite_src = os.path.join(root, f)
                break

assert tflite_src and os.path.exists(tflite_src), \
    f"Could not locate exported .tflite (got {exported!r})"

target = os.path.join(EXPORT_DIR, "pocket_detector_fp16.tflite")
shutil.copy2(tflite_src, target)
print(f"\nFinal TFLite: {target}")
print(f"  size: {os.path.getsize(target) / 1e6:.2f} MB")

# Also export ONNX for parity with the deployed merged_pocket_detector_final.onnx
print("\nExporting ONNX...")
onnx_export = model.export(format="onnx", imgsz=640, opset=17, simplify=True)
onnx_target = os.path.join(EXPORT_DIR, "pocket_detector_final.onnx")
if isinstance(onnx_export, str) and os.path.exists(onnx_export):
    shutil.copy2(onnx_export, onnx_target)
    print(f"ONNX: {onnx_target}  ({os.path.getsize(onnx_target) / 1e6:.2f} MB)")

Exporting TFLite FP16 with NMS...
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/kaggle/working/runs/pocket_detector/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (6.0 MB)
requirements: Ultralytics requirements ['onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 276ms
 Downloaded onnxruntime-gpu
Prepared 2 packages in 3.02s
Installed 2 packages in 14ms
 + onnxruntime-gpu==1.26.0
 + onnxslim==0.1.93

requirements: AutoUpdate success ✅ 3.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 6.0s, saved as '/k

E0000 00:00:1778649478.754179      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778649478.823498      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778649479.378183      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778649479.378209      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778649479.378212      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778649479.378214      23 computation_placer.cc:177] computation placer already registered. Please check linka

requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx2tf>=1.26.3,<1.29.0'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 1.54s
 Downloaded ai-edge-litert
Prepared 5 packages in 257ms
Installed 5 packages in 7ms
 + ai-edge-litert==2.1.4
 + backports-strenum==1.3.1
 + onnx-graphsurgeon==0.6.1
 + onnx2tf==1.28.8
 + sng4onnx==2.0.1

requirements: AutoUpdate success ✅ 2.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


TensorFlow SavedModel: starting export with tensorflow 2.19.0...
Unzipping calibration_image_sample_data_20x128x128x3_float32.npy.zip to /kaggle/working/calibration_image_sample_data_20x128x128x3_float32.npy...: 100% ━━━━━━━━━━━━ 1/1 53.3files/s 0.0s
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...


I0000 00:00:1778649504.024249      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13003 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778649504.029552      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13755 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1778649507.698668      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at '/kaggle/working/runs/pocket_detector/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 300, 6), dtype=tf.float32, name=None)
Captures:
  133771934159248: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133771934158288: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  133771934157328: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  133771934162704: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  133771934163856: TensorSpec(shape=(3, 3, 16, 32), dtype=tf.float32, name=None)
  133771934163472: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  133771934164048: TensorSpec(shape=(1, 1, 32, 32), dtype=tf.float32, name=None)
  133771934159440: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  133771934165008: TensorSpec(shape=(4,), dtype=tf.int64, name=Non

I0000 00:00:1778649514.373992      23 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1778649514.374170      23 single_machine.cc:374] Starting new session
I0000 00:00:1778649514.387479      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13003 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778649514.388914      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13755 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
W0000 00:00:1778649515.313454      23 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778649515.313514      23 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1778649516.218078      23 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 2
I0000 00:00:1778649516.218

TensorFlow SavedModel: export success ✅ 47.3s, saved as '/kaggle/working/runs/pocket_detector/weights/best_saved_model' (29.7 MB)

TensorFlow Lite: starting export with tensorflow 2.19.0...
TensorFlow Lite: export success ✅ 0.0s, saved as '/kaggle/working/runs/pocket_detector/weights/best_saved_model/best_float16.tflite' (6.0 MB)

Export complete (47.6s)
Results saved to /kaggle/working/runs/pocket_detector/weights/best_saved_model/best_float16.tflite
Predict:         yolo predict task=detect model=/kaggle/working/runs/pocket_detector/weights/best_saved_model/best_float16.tflite imgsz=640 half
Validate:        yolo val task=detect model=/kaggle/working/runs/pocket_detector/weights/best_saved_model/best_float16.tflite imgsz=640 data=/kaggle/working/combined_dataset/data.yaml half 
Visualize:       https://netron.app
Export returned: /kaggle/working/runs/pocket_detector/weights/best_saved_model/best_float16.tflite

Final TFLite: /kaggle/working/exports/pocket_detector_fp16.tflite
  size:

## Cell 7 — Smoke-test the exported TFLite on a held-out image

In [8]:
import os, numpy as np, cv2
import tensorflow as tf  # comes preinstalled on Kaggle

interp = tf.lite.Interpreter(
    model_path="/kaggle/working/exports/pocket_detector_fp16.tflite"
)
interp.allocate_tensors()
inp_d, out_d = interp.get_input_details(), interp.get_output_details()
print("Input  :", inp_d[0]["shape"], inp_d[0]["dtype"])
print("Output :", [(o["shape"].tolist(), o["dtype"].__name__)
                   for o in out_d])

# pick a test image
test_imgs = sorted(
    os.path.join(r, f)
    for r, _, files in os.walk("/kaggle/working/combined_dataset/test/images")
    for f in files if f.lower().endswith((".jpg", ".png"))
)
if not test_imgs:
    test_imgs = sorted(
        os.path.join(r, f)
        for r, _, files in os.walk("/kaggle/working/combined_dataset/valid/images")
        for f in files if f.lower().endswith((".jpg", ".png"))
    )
print(f"Smoke-testing on {test_imgs[0]}")

img = cv2.cvtColor(cv2.imread(test_imgs[0]), cv2.COLOR_BGR2RGB)
SZ = int(inp_d[0]["shape"][1])
inp = np.expand_dims((cv2.resize(img, (SZ, SZ)) / 255.0)
                     .astype(inp_d[0]["dtype"]), 0)
interp.set_tensor(inp_d[0]["index"], inp)
interp.invoke()
out = interp.get_tensor(out_d[0]["index"])
print(f"Output tensor shape: {out.shape}")
print(f"Top detections (first 3 rows): {out[0, :3]}")

Input  : [  1 640 640   3] <class 'numpy.float32'>
Output : [([1, 300, 6], 'float32')]
Smoke-testing on /kaggle/working/combined_dataset/test/images/cue__3-17-2-10-_png_jpg.rf.0b759de6531e59d0dbc1f3f5e36d2b32.jpg
Output tensor shape: (1, 300, 6)
Top detections (first 3 rows): [[    0.12419     0.27075     0.85617     0.91296     0.94846           0]
 [    0.63711     0.29089     0.69975      0.4645     0.92017           2]
 [    0.27459     0.29045     0.33717     0.46461     0.91449           2]]


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


## Cell 8 — Generate training_report.md

In [9]:
from datetime import datetime
report = f"""# Pocket Detector Training Report

Generated: {datetime.now()}

## Hyperparameters
- Model: YOLOv8n
- Epochs: 100 (early stop patience 30)
- Batch: 32
- Image size: 640
- AMP: True

## Datasets
- hereliesaz/cue-detat
- diveshcrazy/pool-table-balls-classification

## Classes
{MASTER_CLASSES}

## Validation metrics
- mAP50:    {metrics.box.map50:.4f}
- mAP50-95: {metrics.box.map:.4f}
- Precision: {metrics.box.mp:.4f}
- Recall:    {metrics.box.mr:.4f}

## Exports
- /kaggle/working/exports/pocket_detector_fp16.tflite
- /kaggle/working/exports/pocket_detector_final.onnx

## Deploy locations (in CueDetat repo)
- app/src/main/assets/ml/merged_pocket_detector_final_float16.tflite
- app/src/main/assets/ml/merged_pocket_detector_final.onnx
"""
with open("/kaggle/working/exports/training_report.md", "w") as f:
    f.write(report)
print(report)

# Pocket Detector Training Report

Generated: 2026-05-13 05:18:42.410930

## Hyperparameters
- Model: YOLOv8n
- Epochs: 100 (early stop patience 30)
- Batch: 32
- Image size: 640
- AMP: True

## Datasets
- hereliesaz/cue-detat
- diveshcrazy/pool-table-balls-classification

## Classes
['pool-table', 'pool-table-hole', 'pool-table-side']

## Validation metrics
- mAP50:    0.9913
- mAP50-95: 0.8154
- Precision: 0.9775
- Recall:    0.9678

## Exports
- /kaggle/working/exports/pocket_detector_fp16.tflite
- /kaggle/working/exports/pocket_detector_final.onnx

## Deploy locations (in CueDetat repo)
- app/src/main/assets/ml/merged_pocket_detector_final_float16.tflite
- app/src/main/assets/ml/merged_pocket_detector_final.onnx

